In [21]:
import pandas as pd
import numpy as np

In [22]:
### DEFINE BETA VALUES FOR VDF

beta_vals = {} # TBD: compute values based on growth

beta_template_aux = {
    "Period": ["AM", "MD", "PM", "NT"],
    1: [2.542, 2.545, 2.663, 2.545],
    2: [0.715, 0.708, 0.592, 0.708],
    3: [3.130, 3.133, 3.283, 3.286]
}

beta_vals_aux = pd.DataFrame(beta_template_aux)

beta_vals_aux.set_index("Period", inplace=True)

beta_vals[2025] = beta_vals_aux

In [23]:
### HERE ARE THE MEASURED SPEEDS WE USE AS REFERENCE

measured_speeds_file = r"inputs/measured_speeds.csv"

measured_speeds = pd.read_csv(
    measured_speeds_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

measured_speeds

,1WB,1EB,2WB,2EB,3WB,3EB,4WB,4EB,5WB,5EB,...,9WB,9EB,10WB,10EB,11WB,11EB,12WB,12EB,13WB,13EB
Capacity Factors,,,,,,,,,,,,,,,,,,,,,
Night,62.133376,60.532508,62.546665,62.473888,63.993078,64.432138,67.368247,64.631927,63.220585,64.532924,...,54.330695,61.570981,63.273446,61.775098,63.273446,61.775098,52.526147,54.369840,64.817803,61.316449
AM-Early,60.883671,57.608866,60.898361,61.018102,56.336139,64.597856,62.938370,63.896751,56.626204,63.301478,...,53.284098,59.996419,64.146183,60.967580,64.146183,60.967580,48.885743,54.589016,47.668126,59.629475
AM-Peak,44.362466,25.401137,38.965376,50.541510,35.685073,61.197969,17.738493,59.755295,26.681777,59.316624,...,29.751366,40.876455,64.591643,42.018197,64.591643,42.018197,19.814193,53.330385,33.915135,55.243390
AM-Shoulder,51.231382,30.696695,38.716839,53.707177,41.896190,60.560490,25.623131,58.140287,35.229983,57.885308,...,41.213267,52.776933,64.591643,62.604294,64.591643,62.604294,39.108594,53.218581,40.858394,57.073926
MD,43.493344,55.153130,44.412263,60.684812,56.221637,59.940394,62.120952,45.377404,57.177215,47.344131,...,55.948214,52.295089,64.591643,62.186932,64.591643,62.186932,48.231484,53.804508,44.255045,47.455979
PM-Shoulder,32.068499,55.701287,18.326780,60.385790,35.818657,28.569076,56.865369,26.127821,50.777488,15.310541,...,42.029371,45.588677,22.575720,63.027296,22.575720,63.027296,46.557850,51.287261,56.662112,22.614197
PM-Peak,24.581182,40.928861,18.020722,43.871350,40.135889,19.226766,45.256811,26.625993,52.060494,14.951022,...,35.247353,44.971499,12.637495,49.882566,12.637495,49.882566,45.351579,41.236170,59.577536,19.082191
PM-Late,52.206427,54.179882,52.355064,60.973255,63.887646,60.090240,65.079131,52.150473,61.167601,51.472414,...,52.905898,55.507984,64.591643,63.027296,64.591643,63.027296,52.106928,53.446616,63.003464,43.798176


In [24]:
### DEFINE LOOKUP TABLE FOR BONUS PER PERIOD

lookup_period_file = r"inputs/LookUp_Period.csv"

lookup_period = pd.read_csv(
    lookup_period_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

# Clip y reasignar
lookup_period = lookup_period*0

lookup_period

,Bonus/Mile,4 Periods
Period,,
Night,0.0,
AM-Early,0.0,
AM-Peak,0.0,
AM-Shoulder,0.0,
MD,0.0,
PM-Shoulder,0.0,
PM-Peak,0.0,
PM-Late,0.0,


In [ ]:
### DEFINE SEGMENT PARAMETERS
# Default configuration for time periods in traffic data

#TBD: Make this automatically
period_template = [                 # (Period, Hours/Day, Peak/OP, 4Periods tag)
    ("Night",        8, "OP",   "NT"),
    ("AM-Early",     1, "OP",   "AM"),
    ("AM-Peak",      2, "Peak", "AM"),
    ("AM-Shoulder",  1, "OP",   "AM"),
    ("MD",           5, "OP",   "MD"),
    ("PM-Shoulder",  1, "OP",   "PM"),
    ("PM-Peak",      3, "Peak", "PM"),
    ("PM-Late",      3, "OP",   "PM"),
]

rows = []
years = [2025]

# Default time periods list (for reference)
default_time_periods = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

# Create the base scenario: hour -> time period mapping
hour_to_period = {
    0: "Night",
    1: "Night",
    2: "Night",
    3: "Night",
    4: "Night",
    5: "Night",
    6: "AM-Early",
    7: "AM-Peak",
    8: "AM-Peak",
    9: "AM-Shoulder",
    10: "MD",
    11: "MD",
    12: "MD",
    13: "MD",
    14: "MD",
    15: "PM-Shoulder",
    16: "PM-Peak",
    17: "PM-Peak",
    18: "PM-Peak",
    19: "PM-Late",
    20: "PM-Late",
    21: "PM-Late",
    22: "Night",
    23: "Night"
}

period_to_period = {
    'Evening': 'Night',
    'Evening': 'PM-Late',
    'EarlyAM': 'AM-Early',
    'AM': 'AM-Peak',
    'AM': 'AM-Shoulder',
    'Midday': 'MD',
    'Midday': 'PM-Shoulder',
    'PM': 'PM-Peak'
}

# Define the segments and their parameters

peak_factor = 1 # 1.05 # Peak factor for adjustment at peak hour traffic

hov_percentage = pd.DataFrame({
    'Year' : [2025],
    'HOV percentage' : [0]
})

hov_percentage.set_index('Year', inplace=True)

"""
S1: Chamblee - I 285
S2: I 285 - Jimmy
S3: Jimmy - Indian Trail
S4: Indian Trail -  GA-316
S5: GA-316 - Old Peachtree
S6: Old Peachtree - I-985
S7: I-985 - Hamilton Mill
"""

lengths = [3,3,2.2,2.2,3.4,3.4,4.4,4.4,3,3,4.4,4.4,5.6,5.6]
inscope = [0.95]*14

# Define segment parameters base
seg_params = pd.DataFrame({
    'SegDir':   ["1NB","1SB","2NB","2SB","3NB","3SB","4NB","4SB","5NB","5SB","6NB","6SB","7NB","7SB"],
    'Length':    lengths,
    'Inscope':   inscope,
    'Lanes_GP':  [5,5,6,6,6,6,6,6,5,5,5,5,4,4,3,3,2,2], # We may need to sum the toll lane
    'Lanes_ML':  [1]*14,
    'CapPerLane_GP': [2000]*14,
    'CapPerLane_ML': [1800]*14,
    'Speed_GP':  [65]*4 + [70]*14,
    'Speed_ML':  [70]*14,
    'Alpha_GP':  [1]*14,
    'Beta_GP':   [6]*14,
    'Alpha_ML':  [1]*14,
    'Beta_ML':   [6]*14,
    'Min_Toll_2016': [None]*14,
    'Max_Toll_2016': [None]*14,
    'LanesGP_AM_Peak': [5]*14,
    'LanesGP_PM_Peak': [5]*14,
})

seg_params.set_index('SegDir', inplace=True)

# Compute capacities as lanes * cap per lane
seg_params['Cap_GP'] = seg_params['Lanes_GP'] * seg_params['CapPerLane_GP']
seg_params['Cap_ML'] = seg_params['Lanes_ML'] * seg_params['CapPerLane_ML']

# Compute peak capacities as Alpha * base capacity
seg_params['CapGP_Peak'] = seg_params['Alpha_GP'] * seg_params['Cap_GP']
seg_params['CapML_Peak'] = seg_params['Alpha_ML'] * seg_params['Cap_ML']

# Optional: if you want integer capacities
seg_params[['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']] = seg_params[
    ['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']
].astype(int)

# Preview
seg_params

,Length,Inscope,Lanes_GP,Lanes_ML,CapPerLane_GP,CapPerLane_ML,Speed_GP,Speed_ML,Alpha_GP,Beta_GP,Alpha_ML,Beta_ML,Min_Toll_2016,Max_Toll_2016,LanesGP_AM_Peak,LanesGP_PM_Peak,Cap_GP,Cap_ML,CapGP_Peak,CapML_Peak
SegDir,,,,,,,,,,,,,,,,,,,,
1WB,2.4,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
1EB,2.4,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
2WB,3.2,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
2EB,3.2,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
3WB,3.2,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
3EB,3.2,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
4WB,1.8,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
4EB,1.8,0.65,5,2,2000,1800,65,70,1,6,1,6,None,None,5,5,10000,3600,10000,3600
5WB,3.9,0.65,4,2,2000,1800,65,70,1,6,1,6,None,None,5,5,8000,3600,8000,3600


In [26]:
import numpy as np

def adjusted_cumprod(row, target_year, multiplier):
    years = row.index
    print(row.values)
    factors = 1 + row.values
    
    # Find the index of the target year
    target_idx = list(years).index(target_year)
    
    # Apply multiplier to the target year's factor
    factors[target_idx] *= multiplier
    
    # Calculate cumulative product
    return pd.Series(np.cumprod(factors), index=years)

In [27]:
### IMPORT GROWTHS FOR EACH CLASS
file_path_growths = r"inputs/growths_per_segment.csv"
base_growth_df = pd.read_csv(
    file_path_growths,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

base_growth_df = base_growth_df.iloc[:, 1:]
project_years = base_growth_df.columns[1:].tolist()
base_growth_df.iloc[:, 1:] =  base_growth_df.iloc[:, 1:] + 1

base_growth_df.loc[:, '2032'] *= 1.12

base_growth_df

,SegmentMapped,2025,2026,2027,2028,2029,2030,2031,2032,2033,...,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054
0,S1,1,1.014281,1.014281,1.014281,1.014281,1.016553,1.016553,1.138540,1.016553,...,1.014706,1.014706,1.014706,1.014706,1.014706,1.013360,1.013360,1.013360,1.013360,1.013360
1,S2,1,1.013951,1.013951,1.013951,1.013951,1.016255,1.016255,1.138206,1.016255,...,1.014505,1.014505,1.014505,1.014505,1.014505,1.013242,1.013242,1.013242,1.013242,1.013242
2,S3,1,1.014263,1.014263,1.014263,1.014263,1.016485,1.016485,1.138464,1.016485,...,1.014751,1.014751,1.014751,1.014751,1.014751,1.013492,1.013492,1.013492,1.013492,1.013492
3,S4,1,1.014321,1.014321,1.014321,1.014321,1.016514,1.016514,1.138496,1.016514,...,1.014982,1.014982,1.014982,1.014982,1.014982,1.013747,1.013747,1.013747,1.013747,1.013747
4,S5,1,1.014252,1.014252,1.014252,1.014252,1.016410,1.016410,1.138380,1.016410,...,1.015084,1.015084,1.015084,1.015084,1.015084,1.013887,1.013887,1.013887,1.013887,1.013887
5,S6,1,1.014025,1.014025,1.014025,1.014025,1.016204,1.016204,1.138149,1.016204,...,1.015290,1.015290,1.015290,1.015290,1.015290,1.014162,1.014162,1.014162,1.014162,1.014162
6,S7,1,1.013925,1.013925,1.013925,1.013925,1.016081,1.016081,1.138011,1.016081,...,1.015273,1.015273,1.015273,1.015273,1.015273,1.014151,1.014151,1.014151,1.014151,1.014151
7,S8,1,1.013881,1.013881,1.013881,1.013881,1.016023,1.016023,1.137946,1.016023,...,1.015348,1.015348,1.015348,1.015348,1.015348,1.014225,1.014225,1.014225,1.014225,1.014225
8,S9,1,1.013836,1.013836,1.013836,1.013836,1.015960,1.015960,1.137875,1.015960,...,1.015339,1.015339,1.015339,1.015339,1.015339,1.014214,1.014214,1.014214,1.014214,1.014214
9,S10,1,1.016112,1.016112,1.016112,1.016112,1.016112,1.018178,1.140359,1.018178,...,1.015975,1.015975,1.015975,1.015975,1.015975,1.014326,1.014326,1.014326,1.014326,1.014326


In [28]:
### IMPORT COUNTS AND SEPARATE BY CLASS AND PERIODS

file_path_counts = r"inputs/counts_by_hour_grouped_sorted.csv"
base_counts_df = pd.read_csv(
    file_path_counts,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

# --- Ajustar direcciones ---
base_counts_df["Direction"] = base_counts_df["Direction"].replace({"EB": "EB", "WB": "WB"})

# --- Crear columna Seg/Dir ---
base_counts_df["Seg/Dir"] = base_counts_df["Segment"].astype(str) + base_counts_df["Direction"]

# --- Función para procesar cada clase ---
def process_class(df_class):
    # Convertir a formato largo
    df_long = df_class.melt(
        id_vars=["Seg/Dir", "Segment", "Direction", "Class"],
        value_vars=[str(h) for h in range(24)],
        var_name="Hour",
        value_name="Volume"
    )
    
    # Mapear hora a periodo
    df_long["Hour"] = df_long["Hour"].astype(int)
    df_long["Period"] = df_long["Hour"].map(hour_to_period)
    
    # Agregar por Segment/Direction/Class/Period
    df_period = df_long.groupby(
        ["Seg/Dir", "Segment", "Direction", "Class", "Period"], as_index=False, sort=False
    ).agg({"Volume": "mean"}).round(0)
    
    # Pivot a formato ancho (periodos como columnas)
    period_order = df_period['Period'].unique()
    df_wide = df_period.pivot(
        index=["Seg/Dir", "Segment", "Direction", "Class"],
        columns="Period",
        values="Volume"
    )[period_order].reset_index()
    
    # Mantener solo Seg/Dir como índice
    df_proc = df_wide.drop(columns=["Class", "Direction", "Segment"]).set_index("Seg/Dir")
    
    return df_proc

# --- Separar por clases y procesar ---
dfs_by_class = {}
for cls in base_counts_df["Class"].unique():
    df_cls = base_counts_df[base_counts_df["Class"] == cls].copy()
    dfs_by_class[cls] = process_class(df_cls)


'''
Vehicle Classifications follow FHWA standards:
Lights: FHWA Classes 1-3 [Light Duty Vehicles]
Medium A: Classes 4-5 [Buses and Single Unit 2 axles trucks] 
Medium B: Class 6-7 [Single Unit 3 or 4 axles Trucks]
Heavy A: Classes 8-10 [Single Trailer 3 or more axles trucks]
Heavy B: Classes 11-13 [Combination Trucks Multitrailer Trucks]
'''

# --- Ejemplo de uso ---
df_lights = dfs_by_class["Lights"]
df_mediumA = dfs_by_class["Medium A"]
df_mediumB = dfs_by_class["Medium B"]
df_heavyA = dfs_by_class["Heavy A"]
df_heavyB = dfs_by_class["Heavy B"]

df_lights

Period,Night,AM-Early,AM-Peak,AM-Shoulder,MD,PM-Shoulder,PM-Peak,PM-Late
Seg/Dir,,,,,,,,
S10EB,1171.0,5968.0,6630.0,6140.0,5384.0,5190.0,5030.0,3449.0
S10WB,1117.0,3808.0,5812.0,4579.0,5376.0,6632.0,5996.0,4118.0
S11EB,1171.0,5968.0,6630.0,6140.0,5384.0,5190.0,5030.0,3449.0
S11WB,1117.0,3808.0,5812.0,4579.0,5376.0,6632.0,5996.0,4118.0
S12EB,350.0,1233.0,2037.0,1793.0,1924.0,2656.0,3087.0,1740.0
S12WB,369.0,2117.0,2712.0,2522.0,1967.0,1863.0,1897.0,1331.0
S13EB,1732.0,3683.0,5026.0,4180.0,5587.0,5991.0,6110.0,5355.0
S13WB,1258.0,5698.0,5725.0,4991.0,4016.0,4146.0,3785.0,2514.0
S1EB,1372.0,6352.0,6344.0,5451.0,5650.0,5767.0,6156.0,4060.0


In [29]:
# --- Lista de periodos según tus columnas ---
period_cols = ["Night","AM-Early","AM-Peak","AM-Shoulder","MD","PM-Shoulder","PM-Peak","PM-Late"]

# Diccionario de dataframes por clase
class_dfs = {
    "Lights": df_lights,
    "Medium A": df_mediumA,
    "Medium B": df_mediumB,
    "Heavy A": df_heavyA,
    "Heavy B": df_heavyB
}

projected_long_by_class = {}

for cls_name, df_class in class_dfs.items():
    df = df_class.copy()
    
    # Resetear índice Seg/Dir y extraer Segment y Direction
    df = df.reset_index()
    df["Segment"] = df["Seg/Dir"].str.extract(r"(\d+)")[0]    # solo los números
    df["Direction"] = df["Seg/Dir"].str.extract(r"([A-Z]+)")[0]  # solo las letras
    df["Class"] = cls_name
    
    # Melt usando las columnas de periodos
    df_long = df.melt(
        id_vars=["Seg/Dir","Segment","Direction","Class"],
        value_vars=period_cols,
        var_name="Period",
        value_name="AADT "+str(df["Class"][0])
    )
    
    # Normalizar SegDir (opcional)
    df_long["SegDir"] = df_long["Seg/Dir"].str.strip().str.upper().str.lstrip("S")
    
    projected_long_by_class[cls_name] = df_long

# Ejemplo: ver Lights
projected_long_lights_df = projected_long_by_class["Lights"]
projected_long_mediumA_df = projected_long_by_class["Medium A"]
projected_long_mediumB_df = projected_long_by_class["Medium B"]
projected_long_heaviesA_df = projected_long_by_class["Heavy A"]
projected_long_heaviesB_df = projected_long_by_class["Heavy B"]
projected_long_lights_df


,Seg/Dir,Segment,Direction,Class,Period,AADT Lights,SegDir
0,S10EB,10,S,Lights,Night,1171.0,10EB
1,S10WB,10,S,Lights,Night,1117.0,10WB
2,S11EB,11,S,Lights,Night,1171.0,11EB
3,S11WB,11,S,Lights,Night,1117.0,11WB
4,S12EB,12,S,Lights,Night,350.0,12EB
...,...,...,...,...,...,...,...
203,S7WB,7,S,Lights,PM-Late,2537.0,7WB
204,S8EB,8,S,Lights,PM-Late,3600.0,8EB
205,S8WB,8,S,Lights,PM-Late,3223.0,8WB
206,S9EB,9,S,Lights,PM-Late,2492.0,9EB


In [30]:
rows = []

for year in years:
    for seg in seg_params.index:  # e.g., "1NB", "1SB", etc.
        seg_data = seg_params.loc[seg]
        # Extraer parte numérica y dirección
        seg_numeric = ''.join(filter(str.isdigit, seg))  # e.g., "10"
        direction = seg[len(seg_numeric):]       
        for p, hrs, peak, tag in period_template:
            rows.append({
                "Year": year,
                "SegDir": seg,
                "Segment": seg_numeric,        
                "Direction": direction,     
                "Period": p,
                "Hours/Day": hrs,
                "Peak": peak,
                "4Periods": tag,

                # Parámetros técnicos
                "Length": seg_data["Length"],
                "Speed GP": seg_data["Speed_GP"],
                "Capacity GP": seg_data["CapPerLane_GP"] * seg_data["Lanes_GP"],
                "Alpha GP": seg_data["Alpha_GP"],
                "Beta GP": seg_data["Beta_GP"],
                "Speed ML": seg_data["Speed_ML"],
                "Capacity ML": seg_data["CapPerLane_ML"] * seg_data["Lanes_ML"],
                "Alpha ML": seg_data["Alpha_ML"],
                "Beta ML": seg_data["Beta_ML"],
                "MinToll": 0.5,
                "MinCapture": 0
            })

# --- plantilla base ---
first_model_df = pd.DataFrame(rows)

# --- merge para todas las clases ---
for cls_name, df_proj in projected_long_by_class.items():
    proj_merge_df = df_proj[["SegDir", "Period", f"AADT {cls_name}"]].copy()
    proj_merge_df.rename(columns={"AADT": f"AADT {cls_name}"}, inplace=True)

    first_model_df = first_model_df.merge(
        proj_merge_df,
        on=["SegDir", "Period"],
        how="left"
    )

# --- 1. Reshape growths a formato largo ---
growths_long = base_growth_df.melt(
    id_vars="SegmentMapped",
    var_name="Year",
    value_name="AnnualGrowth"
).copy()
growths_long["Year"] = growths_long["Year"].astype(int)

# --- 2. Calcular crecimiento acumulado desde 2025 ---
# Ordenamos por año y aplicamos cumprod
growths_long = growths_long.sort_values(["SegmentMapped", "Year"])
growths_long["GrowthFactor"] = (growths_long["AnnualGrowth"]).groupby(growths_long["SegmentMapped"]).cumprod()

# Ahora GrowthFactor(y) = factor acumulado 2025→y

# --- 3. Preparar plantilla ---
fm = first_model_df.copy()
fm["Year"] = fm["Year"].astype(int)
fm["Segment"] = fm["Segment"].astype(str).str.replace(r"^S", "", regex=True)
fm["SegmentMapped"] = "S" + fm["Segment"].astype(str)

# --- 4. Merge GrowthFactor ---
fm = fm.merge(
    growths_long[["SegmentMapped", "Year", "GrowthFactor"]],
    on=["SegmentMapped", "Year"],
    how="left"
)

fm["GrowthFactor"] = fm["GrowthFactor"].fillna(1.0)

# --- 5. Aplicar GrowthFactor a todas las clases ---
for cls_name in projected_long_by_class.keys():
    col = f"AADT {cls_name}"
    if col in fm.columns:
        fm[col] = (fm[col].fillna(0) * fm["GrowthFactor"]).round(1)

# --- 6. Limpieza ---
fm = fm.drop(columns=["SegmentMapped"])   # opcional

first_model_df = fm

first_model_df


,Year,SegDir,Segment,Direction,Period,Hours/Day,Peak,4Periods,Length,Speed GP,...,Alpha ML,Beta ML,MinToll,MinCapture,AADT Lights,AADT Medium A,AADT Medium B,AADT Heavy A,AADT Heavy B,GrowthFactor
0,2025,1WB,1,WB,Night,8,OP,NT,2.4,65,...,1,6,0.5,0,1282.0,70.0,4.0,257.0,5.0,1.0
1,2025,1WB,1,WB,AM-Early,1,OP,AM,2.4,65,...,1,6,0.5,0,5154.0,240.0,12.0,327.0,1.0,1.0
2,2025,1WB,1,WB,AM-Peak,2,Peak,AM,2.4,65,...,1,6,0.5,0,6641.0,256.0,36.0,257.0,2.0,1.0
3,2025,1WB,1,WB,AM-Shoulder,1,OP,AM,2.4,65,...,1,6,0.5,0,5699.0,202.0,40.0,532.0,6.0,1.0
4,2025,1WB,1,WB,MD,5,OP,MD,2.4,65,...,1,6,0.5,0,6069.0,220.0,42.0,569.0,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,2025,13EB,13,EB,AM-Shoulder,1,OP,AM,3.3,65,...,1,6,0.5,0,4180.0,225.0,47.0,427.0,10.0,1.0
204,2025,13EB,13,EB,MD,5,OP,MD,3.3,65,...,1,6,0.5,0,5587.0,207.0,38.0,391.0,5.0,1.0
205,2025,13EB,13,EB,PM-Shoulder,1,OP,PM,3.3,65,...,1,6,0.5,0,5991.0,146.0,12.0,256.0,6.0,1.0
206,2025,13EB,13,EB,PM-Peak,3,Peak,PM,3.3,65,...,1,6,0.5,0,6110.0,153.0,8.0,164.0,3.0,1.0


In [ ]:
first_model_df["Capacity GP"] = first_model_df.apply(
    lambda row: seg_params.loc[row["SegDir"], 'Cap_GP'],
    axis=1
)

first_model_df["B1"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 1],
    axis=1
)

first_model_df["B2"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 2],
    axis=1
)

# Here we load th value of the counts and we multiply the peak hour values by a constant
lights_w = 1

heavies_w = 3
heavies_w_toll = 3
heavies_w_vot = 4

medium_A_w = 3 # 1.5 # TBD: Maybe try 2.5 or 2.75 for every pce value
medium_A_w_toll = 4
medium_A_w_vot = 4

medium_B_w = 3 # 2.75
medium_B_w_toll = 5
medium_B_w_vot = 4

heavy_A_w = 3 # 2.75
heavy_A_w_toll = 3
heavy_A_w_vot = 3

heavy_B_w = 3
heavy_B_w_toll = 3
heavy_B_w_vot = 3

# 2. Compute TotalLights
first_model_df["TotalLights"] = first_model_df["AADT Lights"] 

first_model_df["TotalMediumA"] = first_model_df["AADT Medium A"]

first_model_df["TotalMediumB"] = first_model_df["AADT Medium B"]

first_model_df["TotalHeavyA"] = first_model_df["AADT Heavy A"]

first_model_df["TotalHeavyB"] = first_model_df["AADT Heavy B"]

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["Corridor PCE pre-fix"] = first_model_df.apply(
    lambda row: row["TotalLights"] * lights_w + row["TotalMediumA"] * medium_A_w + row["TotalMediumB"] * medium_B_w + row["TotalHeavyA"] * heavy_A_w + row["TotalHeavyB"] * heavy_B_w,
    axis=1
)

first_model_df["Corridor PCE"] = first_model_df["Corridor PCE pre-fix"] # To anulate the suppression


first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalLights'] *= peak_factor

first_model_df["HOV3"] = first_model_df.apply(
    lambda row: row["TotalLights"] * hov_percentage.loc[row['Year']]['HOV percentage'],
    axis=1
)

first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumB'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyB'] *= peak_factor

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["InScopeLights"] = first_model_df.apply(
    lambda row: (row["TotalLights"] - row["HOV3"]) * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyA"] = first_model_df.apply(
    lambda row: row["TotalHeavyA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyB"] = first_model_df.apply(
    lambda row: row["TotalHeavyB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeVeh"] = first_model_df.apply(
    lambda row: row["TotalVeh"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

# first_model_df.to_csv('model_test.csv')

In [32]:
def get_speed(row):

    max_VC = 1.2  # TBD: check if we need to change this value
    ETC_discount = 0.15


    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speedGP =  row["Speed GP"] / (1 + row["Alpha GP"] * ((gp_pce / row["Capacity GP"]) ** row["Beta GP"]))

    timeGP = 60 * row["Length"] / speedGP

    gp_vc = gp_pce / row["Capacity GP"]

    return pd.Series([speedGP, timeGP], index=["Speed GP","Time GP"])

In [33]:
first_model_df[["Speed GP Real", "Time GP"]] = first_model_df.apply(
    get_speed, axis=1, result_type='expand'
)

first_model_df.to_csv('model_test.csv')

In [34]:
def get_pce(row):

    measured_speed = measured_speeds.loc[row['Period']][row['SegDir']]

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speed_rate = row['Speed GP'] / measured_speed

    if speed_rate < 1:

        measured_speed = row['Speed GP'] - 0.01

    gp_pce_aux = row["Capacity GP"] * ((row['Speed GP'] / (measured_speed * row['Alpha GP']) - 1 / row['Alpha GP']) ** (1 / row['Beta GP']))

    pce_factor = gp_pce_aux / gp_pce

    pce_factor = np.clip(pce_factor, a_min = 1, a_max = 1.2)

    if (row["Period"] == "AM-Peak") or (row["Period"] == "PM-Peak"):
        return pce_factor
    else:
        return 1

    

In [35]:
first_model_df["PCE Factor"] = first_model_df.apply(
    lambda row: get_pce(row),
    axis=1
)

first_model_df.to_csv('model_test.csv')

In [36]:
cap_factor = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="PCE Factor"
)

cap_factor.to_csv('inputs/pce_factors.csv')

cap_factor

SegDir,10EB,10WB,11EB,11WB,12EB,12WB,13EB,13WB,1EB,1WB,...,5EB,5WB,6EB,6WB,7EB,7WB,8EB,8WB,9EB,9WB
Period,,,,,,,,,,,,,,,,,,,,,
AM-Early,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
AM-Peak,1.2,1.0,1.2,1.0,1.2,1.2,1.123499,1.2,1.2,1.061312,...,1.2,1.2,1.0,1.2,1.2,1.2,1.2,1.2,1.2,1.2
AM-Shoulder,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
MD,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Night,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PM-Late,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PM-Peak,1.2,1.2,1.2,1.2,1.2,1.2,1.200000,1.2,1.2,1.200000,...,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2
PM-Shoulder,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [37]:
first_model_df["TotalLights"] = first_model_df.apply(
    lambda row: row["TotalLights"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalMediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalMediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalHeavyA"] = first_model_df.apply(
    lambda row: row["TotalHeavyA"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalHeavyB"] = first_model_df.apply(
    lambda row: row["TotalHeavyB"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

In [38]:
def get_capacity(row):

    measured_speed = measured_speeds.loc[row['Period']][row['SegDir']]

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speed_rate = row['Speed GP'] / measured_speed

    if speed_rate < 1:

        measured_speed = row['Speed GP'] - 0.01

    capacity = gp_pce / ((row['Speed GP'] / (measured_speed * row['Alpha GP']) - 1 / row['Alpha GP']) ** (1 / row['Beta GP']))

    capacity_factor = capacity / row["Capacity GP"]

    return capacity_factor

In [39]:
first_model_df["Capacity Factor"] = first_model_df.apply(
    lambda row: get_capacity(row),
    axis=1
)

cap_factor = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Capacity Factor"
)

cap_factor.to_csv('inputs/capacity_factors.csv')

cap_factor

SegDir,10EB,10WB,11EB,11WB,12EB,12WB,13EB,13WB,1EB,1WB,...,5EB,5WB,6EB,6WB,7EB,7WB,8EB,8WB,9EB,9WB
Period,,,,,,,,,,,,,,,,,,,,,
AM-Early,1.053571,0.917183,1.053571,0.917183,0.556797,0.558325,0.746363,0.749026,1.163080,1.080114,...,0.600151,0.912238,1.035633,1.021160,1.722347,0.672607,0.866359,0.756945,0.594868,0.517444
AM-Peak,0.953020,1.504388,0.953020,1.504388,0.850728,0.542512,1.000000,0.783253,0.913589,1.000000,...,0.860812,0.702423,1.095464,0.926281,0.939658,0.567716,0.831908,0.577210,0.661806,0.518071
AM-Shoulder,1.205505,1.245556,1.205505,1.245556,0.673481,0.565673,0.876434,0.634580,0.796319,1.000713,...,0.733738,0.570797,0.952736,0.758015,0.780321,0.539076,0.833353,0.616032,0.627448,0.479470
MD,1.047718,1.419738,1.047718,1.419738,0.765094,0.525518,0.886476,0.531218,1.044388,0.964518,...,0.654904,0.768982,0.951235,0.856014,0.657175,0.652965,0.808866,0.869648,0.603982,0.578049
Night,0.217064,0.234194,0.217064,0.234194,0.151000,0.114820,0.410503,0.385177,0.348486,0.382379,...,0.262043,0.232535,0.579731,0.960315,0.299985,0.371481,0.342426,0.374824,0.252926,0.195042
PM-Late,0.652323,1.015561,0.652323,1.015561,0.565385,0.377775,0.691000,0.477301,0.662492,0.772884,...,0.508534,0.517875,0.710996,0.792961,0.508757,0.571276,0.595453,0.630157,0.451264,0.375856
PM-Peak,0.789630,0.602964,0.789630,0.602964,0.829182,0.517447,0.735383,0.738409,0.971846,0.768986,...,0.436236,0.760849,0.547601,0.957258,0.446362,0.764745,0.714652,0.903807,0.686593,0.499923
PM-Shoulder,1.018565,0.642111,1.018565,0.642111,0.804191,0.452592,0.653017,0.659788,1.010864,0.624530,...,0.440301,0.673153,0.560411,0.743063,0.412702,0.609127,0.627888,0.750775,0.586377,0.473230


In [40]:
period_order = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

first_model_df["Total Corridor"] = first_model_df.apply(
    lambda row: row["Corridor PCE"] * row["Hours/Day"],
    axis=1
)


first_model_df["Period"] = pd.Categorical(
    first_model_df["Period"],
    categories=period_order,
    ordered=True
)

corridor_pce = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Total Corridor"
)

corridor_pce.to_csv('corridor_vals.csv')